In [ ]:
import re
import os
import time
import math
import numpy as np
import subprocess

In [ ]:
def parse_games(text):
    chunks = re.split(r'(?=\[Event\s+")', text.strip())
    games = []

    for chunk in chunks:
        if not chunk.strip():
            continue

        headers = dict(re.findall(r'\[(\w+)\s+"(.*?)"\]', chunk))
        moves = re.findall(r'\b[a-i][0-9][a-i][0-9]\b', chunk.lower())

        games.append({
            "headers": headers,
            "moves": moves
        })

    return games

def load_games_from_files(file_paths):
    all_games = []

    for path in file_paths:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        games = parse_games(text)
        print(f"{path}: {len(games)} games")
        all_games.extend(games)

    print("Total games loaded:", len(all_games))
    return all_games

In [ ]:
FILES = "abcdefghi"
FILE_TO_X = {c: i for i, c in enumerate(FILES)}
X_TO_FILE = {i: c for i, c in enumerate(FILES)}

PIECE_TO_CHANNEL = {
    "rk": 0, "rr": 1, "rh": 2, "re": 3, "ra": 4, "rc": 5, "rp": 6,
    "bk": 7, "br": 8, "bh": 9, "be": 10, "ba": 11, "bc": 12, "bp": 13
}

def initial_board():
    b = [["." for _ in range(9)] for _ in range(10)]

    # Red
    b[0] = ["rr","rh","re","ra","rk","ra","re","rh","rr"]
    b[2][1], b[2][7] = "rc", "rc"
    for x in [0,2,4,6,8]:
        b[3][x] = "rp"

    # Black
    b[9] = ["br","bh","be","ba","bk","ba","be","bh","br"]
    b[7][1], b[7][7] = "bc", "bc"
    for x in [0,2,4,6,8]:
        b[6][x] = "bp"

    return b

def apply_move(board, move):
    sx, sy = FILE_TO_X[move[0]], int(move[1])
    dx, dy = FILE_TO_X[move[2]], int(move[3])

    board[dy][dx] = board[sy][sx]
    board[sy][sx] = "."

def board_to_tensor(board, side_to_move):
    x = np.zeros((15, 10, 9), dtype=np.float32)

    for y in range(10):
        for col in range(9):
            p = board[y][col]
            if p != ".":
                x[PIECE_TO_CHANNEL[p], y, col] = 1.0

    x[14, :, :] = 1.0 if side_to_move == "red" else 0.0
    return x

In [ ]:
def move_to_id(move):
    sx, sy = FILE_TO_X[move[0]], int(move[1])
    dx, dy = FILE_TO_X[move[2]], int(move[3])

    from_idx = sy * 9 + sx
    to_idx = dy * 9 + dx
    return from_idx * 90 + to_idx

def id_to_move(move_id):
    from_idx = move_id // 90
    to_idx = move_id % 90

    sx = from_idx % 9
    sy = from_idx // 9
    dx = to_idx % 9
    dy = to_idx // 9

    return f"{X_TO_FILE[sx]}{sy}{X_TO_FILE[dx]}{dy}"

In [ ]:
PIKAFISH_PATH = "/Users/anguschua/Pikafish/src/pikafish"   # change if needed
class PikafishEngine:
    def __init__(self, path, threads=1, hash_mb=64):
        self.p = subprocess.Popen(
            [path],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        self.send("uci")
        self._wait_for("uciok")

        self.send(f"setoption name Threads value {threads}")
        self.send(f"setoption name Hash value {hash_mb}")

        self.send("isready")
        self._wait_for("readyok")

    def send(self, cmd):
        self.p.stdin.write(cmd + "\n")
        self.p.stdin.flush()

    def _wait_for(self, token):
        while True:
            line = self.p.stdout.readline().strip()
            if token in line:
                return line

    def new_game(self):
        self.send("ucinewgame")
        self.send("isready")
        self._wait_for("readyok")

    def bestmove_from_prefix(self, moves_prefix, depth=2):
        self.new_game()

        if moves_prefix:
            cmd = "position startpos moves " + " ".join(moves_prefix)
        else:
            cmd = "position startpos"

        self.send(cmd)
        self.send(f"go depth {depth}")

        bestmove = None

        while True:
            line = self.p.stdout.readline().strip()

            if line.startswith("bestmove"):
                parts = line.split()
                if len(parts) >= 2:
                    bestmove = parts[1]
                break

        return bestmove

    def close(self):
        try:
            self.send("quit")
        except:
            pass
        self.p.terminate()


In [ ]:
def build_policy_dataset_from_pikafish(games, engine_path, depth=8, threads=2, hash_mb=64):
    engine = PikafishEngine(engine_path, threads=threads, hash_mb=hash_mb)

    X = []
    y = []

    total_positions = sum(len(g["moves"]) for g in games)
    done = 0

    try:
        for gi, game in enumerate(games):
            board = initial_board()
            side = "red"
            moves_prefix = []

            for move in game["moves"]:
                if done % 100 == 0:
                    print(f"{done}/{total_positions}")

                # current board tensor
                x_tensor = board_to_tensor(board, side)

                # Pikafish best move from this position
                bestmove = engine.bestmove_from_prefix(moves_prefix, depth=depth)

                # sometimes engine might return "(none)" in weird positions
                if bestmove is not None and re.fullmatch(r"[a-i][0-9][a-i][0-9]", bestmove):
                    X.append(x_tensor)
                    y.append(move_to_id(bestmove))

                # advance to next real game position
                apply_move(board, move)
                moves_prefix.append(move)
                side = "black" if side == "red" else "red"
                done += 1
    finally:
        engine.close()

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


In [ ]:
file_paths = ["Liu_Dahua_1125_UCI_games.txt", "Hu_Ronghua_1391_UCI_games.txt",]

games = load_games_from_files(file_paths)

In [ ]:
X_pol, y_pol = build_policy_dataset_from_pikafish(
    games,
    engine_path=PIKAFISH_PATH,
    depth=8,
    threads=2,
    hash_mb=64,
)

print("X shape:", X_pol.shape)
print("y shape:", y_pol.shape)
print("first 10 labels:", y_pol[:10])

In [ ]:
np.savez_compressed("pikafish_policy_dataset.npz", X=X_pol, y=y_pol)
print("saved xiangqi_pikafish_policy_dataset.npz")